# MCP server 6 — Vibration diagnostics

Run useful DSP-domain calculations without a database, then show where telemetry joins the flow.

**Tutorial contract:** run cells from top to bottom. Every external dependency is checked before use,
outputs go under `artifacts/kdd_tutorial/`, and no credential value is printed.


## Goal

Calculate bearing frequencies and ISO severity through MCP.

**Requires:** NumPy/SciPy; CouchDB only for real vibration telemetry


In [ ]:
from pathlib import Path
import json, os, sys

def find_repo(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "servers").exists():
            return candidate
    raise RuntimeError("Open this notebook from inside the AssetOpsBench repository.")

REPO = find_repo()
ARTIFACTS = REPO / "artifacts" / "kdd_tutorial"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
print("repo:", REPO)
print("python:", sys.version.split()[0])


In [ ]:
# Load environment variables from .env file in the repository root
from dotenv import load_dotenv

def find_repo(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src").is_dir():
            return candidate
    return None
    
repo = find_repo()
if repo is None:
    raise RuntimeError("Open this notebook from inside the AssetOpsBench repository.")
ENV_FILE = repo / ".env"
if not ENV_FILE.exists():
    raise RuntimeError(f"Missing {ENV_FILE}. Complete 00_environment_setup.ipynb first.")
load_dotenv(ENV_FILE, override=True)
print("environment source:", ENV_FILE)

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

ENTRY_POINTS = {
    "iot": "iot-mcp-server", "utilities": "utilities-mcp-server",
    "fmsr": "fmsr-mcp-server", "wo": "wo-mcp-server",
    "tsfm": "tsfm-mcp-server", "vibration": "vibration-mcp-server",
}

async def mcp_session(server, operation, tool_name=None, arguments=None):
    params = StdioServerParameters(
        command="uv",
        args=["run", "--directory", str(REPO), ENTRY_POINTS[server]],
        cwd=str(REPO),
    )
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            if operation == "list":
                return await session.list_tools()
            return await session.call_tool(tool_name, arguments or {})

def text_result(result):
    text = "\n".join(getattr(item, "text", str(item)) for item in result.content)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return text

async def list_tools(server):
    response = await mcp_session(server, "list")
    return [{"name": t.name, "description": t.description, "schema": t.inputSchema} for t in response.tools]

async def call_tool(server, name, **arguments):
    return text_result(await mcp_session(server, "call", name, arguments))


## 1. Discover the live MCP contract

This starts the real stdio server and asks it for its tool schemas.


In [ ]:
tools = await list_tools("vibration")
[(t["name"], list(t["schema"].get("properties", {}))) for t in tools]


## 2. Credential-free bearing knowledge


In [ ]:
bearings = await call_tool("vibration", "list_known_bearings")
bearings


In [ ]:
freqs = await call_tool("vibration", "calculate_bearing_frequencies", rpm=1800, n_balls=9, ball_diameter_mm=7.938, pitch_diameter_mm=38.5, contact_angle_deg=0, bearing_name="6205")
freqs


In [ ]:
severity = await call_tool("vibration", "assess_vibration_severity", rms_velocity_mm_s=4.5, machine_group="group2")
severity


## 3. Optional data-backed path

With seeded CouchDB, call `get_vibration_data` first; use its returned `data_id` for FFT, envelope, and diagnosis.


In [ ]:
import socket
from urllib.parse import urlparse

def tcp_reachable(url, timeout=1.0):
    parsed = urlparse(url)
    host, port = parsed.hostname or "localhost", parsed.port or 5984
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False

COUCHDB_URL = os.getenv("COUCHDB_URL", "http://localhost:5984")
print("CouchDB reachable:", tcp_reachable(COUCHDB_URL), "at", COUCHDB_URL)


## Takeaway

You exercised the server through MCP JSON-RPC over stdio—the same boundary the agents use.
